In [1]:
from pathlib import Path
import pandas as pd
import re
import unicodedata

DATA_DIR = Path('../training_datasets')

S1_PATH = DATA_DIR / 'train_source1.tsv'
S2_PATH = DATA_DIR / 'train_source2.tsv'
S3_PATH = DATA_DIR / 'train_source3.tsv'
GT_PATH = DATA_DIR / 'train_ground_truth.tsv'

CHUNK_SIZE = 100_000

# Business Name Normalization

In [10]:
HONORIFICS = {'mr', 'mrs', 'ms', 'dr'}          # sri / smt removed

COMPOUND_SUFFIXES = {
    ('private', 'limited'),
    ('pvt', 'ltd'),
    ('pvt', 'limited'),
}

SUFFIXES = {
    'inc', 'incorporated',
    'ltd', 'limited',
    'llc',
    'corp', 'corporation',
    'co', 'company',
    'pvt',
    'llp',
    'plc'
}

DOTTED_SUFFIX_MAP = {
    r'\bl\.?\s*l\.?\s*c\.?\b': 'llc',
    r'\bp\.?\s*l\.?\s*c\.?\b': 'plc',
    r'\bl\.?\s*t\.?\s*d\.?\b': 'ltd',
    r'\binc\.?\b': 'inc',
    r'\bcorp\.?\b': 'corp',
    r'\bco\.?\b': 'co',
}

def normalize_name(name: str) -> str:
    if not isinstance(name, str) or not name.strip():
        return ''

    # 1. NFKC
    s = unicodedata.normalize('NFKC', name)

    # 2. lowercase
    s = s.lower()

    # 3. dotted legal forms → canonical
    for pattern, replacement in DOTTED_SUFFIX_MAP.items():
        s = re.sub(pattern, replacement, s)

    # 4. & → and (track whether we did it)
    ampersand_replaced = '&' in s
    s = s.replace('&', ' and ')

    # 5. punctuation → spaces
    s = re.sub(r'[^\w\s]', ' ', s)

    # 6. collapse whitespace
    s = re.sub(r'\s+', ' ', s).strip()

    tokens = s.split()

    # 7. remove leading honorifics
    while tokens and tokens[0] in HONORIFICS:
        tokens.pop(0)

    # 8. remove compound legal suffixes first
    for compound in COMPOUND_SUFFIXES:
        n = len(compound)
        if len(tokens) >= n and tuple(tokens[-n:]) == compound:
            tokens = tokens[:-n]
            break

    # 9. remove ordinary trailing suffixes
    while tokens and tokens[-1] in SUFFIXES:
        tokens.pop()

    # 10. remove dangling "and" only if we created it
    if ampersand_replaced and tokens and tokens[-1] == 'and':
        tokens.pop()

    return ' '.join(tokens)

In [11]:
test_names = [
    "Dr. Sharma & Co.",
    "Sharma & Company",
    "SRI RAM PRIVATE LIMITED",
    "SRI RAM PVT LTD",
    "Tata Motors Ltd.",
    "Apple Inc.",
    "Microsoft Corporation",
    "ABC L.L.C.",
    "ABC L T D",
    "ABC P.L.C.",
    "MÜLLER & SÖHNE",
    "Dr.   Sharma   &   Co.",
    "Sharma and Sons",
    "Sri Lakshmi Traders",
]

for name in test_names:
    print(f"{name!r:40s} → {normalize_name(name)!r}")

'Dr. Sharma & Co.'                       → 'sharma'
'Sharma & Company'                       → 'sharma'
'SRI RAM PRIVATE LIMITED'                → 'sri ram'
'SRI RAM PVT LTD'                        → 'sri ram'
'Tata Motors Ltd.'                       → 'tata motors'
'Apple Inc.'                             → 'apple'
'Microsoft Corporation'                  → 'microsoft'
'ABC L.L.C.'                             → 'abc'
'ABC L T D'                              → 'abc'
'ABC P.L.C.'                             → 'abc'
'MÜLLER & SÖHNE'                         → 'müller and söhne'
'Dr.   Sharma   &   Co.'                 → 'sharma'
'Sharma and Sons'                        → 'sharma and sons'
'Sri Lakshmi Traders'                    → 'sri lakshmi traders'


# Business Address Normalization

In [17]:
def normalize_address(addr: str) -> str:
    if not isinstance(addr, str) or not addr.strip():
        return ''
    s = unicodedata.normalize('NFKC', addr)
    s = s.lower()
    s = re.sub(r'[^\w\s]', ' ', s)       # punctuation → space
    s = re.sub(r'\s+', ' ', s).strip()   # collapse whitespace
    return s


def extract_zip_pin(addr: str, country: str) -> str:
    """Extract from the *raw* address, not the cleaned one."""
    if not isinstance(addr, str):
        return ''
    if country == 'US':
        m = re.search(r'\b(\d{5})(?:-\d{4})?\b', addr)
    elif country == 'India':
        m = re.search(r'\b(\d{6})\b', addr)
    else:
        return ''
    return m.group(1) if m else ''


def extract_leading_number(addr: str, country: str = '') -> str:
    if not isinstance(addr, str):
        return ''

    m = re.match(r'^\s*(\d+[A-Za-z]?)', addr)
    if not m:
        return ''

    num = m.group(1)

    # If it looks like a full ZIP/PIN, don't treat it as a street number
    if country == 'US' and re.fullmatch(r'\d{5}', num):
        return ''
    if country == 'India' and re.fullmatch(r'\d{6}', num):
        return ''

    return num

In [20]:
test_cases = [
    # (raw_address, country, expected_clean, expected_zip_pin, expected_leading)
    ("123 Main Street, New York, NY 10001",          "US",    "123 main street new york ny 10001", "10001", "123"),
    ("123 Main Street, New York, NY 10001-1234",     "US",    "123 main street new york ny 10001 1234", "10001", "123"),
    ("560034, Bangalore, Karnataka",                 "India", "560034 bangalore karnataka", "560034", ""),
    ("45A MG Road, Bengaluru 560001",                "India", "45a mg road bengaluru 560001", "560001", "45A"),
    ("Plot No. 12, Sector 5",                        "India", "plot no 12 sector 5", "", ""),
    ("Empire State Building, New York",              "US",    "empire state building new york", "", ""),
    ("##120 WOOD THRUSH LN, MOORESVILLE, NC",        "US",    "120 wood thrush ln mooresville nc", "", ""),
    ("No.2A, Saravana Signature Suites, Coimbatore", "India", "no 2a saravana signature suites coimbatore", "", ""),
    ("10001 Broadway, New York",                     "US",    "10001 broadway new york", "10001", ""),
    ("12B, 1st Floor, Andheri East, Mumbai 400069",  "India", "12b 1st floor andheri east mumbai 400069", "400069", "12B"),
    ("",                                             "US",    "", "", ""),
    (None,                                           "India", "", "", ""),
]

print(f"{'Raw':<55} | {'Clean':<45} | {'ZIP/PIN':<8} | {'Lead'}")
print("-" * 120)

for raw, country, exp_clean, exp_zip, exp_lead in test_cases:
    clean = normalize_address(raw)
    z = extract_zip_pin(raw or '', country)
    lead = extract_leading_number(raw or '', country)

    # simple visual check
    ok = (clean == exp_clean) and (z == exp_zip) and (lead == exp_lead)
    status = "✓" if ok else "✗"

    print(f"{status} {str(raw)[:53]:<53} | {clean[:43]:<43} | {z:<8} | {lead}")

Raw                                                     | Clean                                         | ZIP/PIN  | Lead
------------------------------------------------------------------------------------------------------------------------
✓ 123 Main Street, New York, NY 10001                   | 123 main street new york ny 10001           | 10001    | 123
✓ 123 Main Street, New York, NY 10001-1234              | 123 main street new york ny 10001 1234      | 10001    | 123
✓ 560034, Bangalore, Karnataka                          | 560034 bangalore karnataka                  | 560034   | 
✓ 45A MG Road, Bengaluru 560001                         | 45a mg road bengaluru 560001                | 560001   | 45A
✓ Plot No. 12, Sector 5                                 | plot no 12 sector 5                         |          | 
✓ Empire State Building, New York                       | empire state building new york              |          | 
✓ ##120 WOOD THRUSH LN, MOORESVILLE, NC             